# 12 — End-to-End Conversation Scenarios

Full integration tests that run realistic multi-turn conversations through the Router and validate the complete pipeline:

1. Gift search with no constraints  
2. Allergen-aware search (critic loop activated)  
3. Delivery feasibility check  
4. Mixed intent (search + logistics + preference update)  
5. Multi-turn session (profile persists across turns)  
6. Latency measurement  

> **Requires**: all credentials in `.env` + catalog ingested into Qdrant.

In [ ]:
import sys, os, time, json, uuid
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'))

assert os.getenv('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY missing'
assert os.getenv('QDRANT_URL'), 'QDRANT_URL missing'
assert os.getenv('QDRANT_API_KEY'), 'QDRANT_API_KEY missing'
print('All env vars present: OK')

In [ ]:
from agents.router import Router

def run_conversation(router, message, label=''):
    """Run a single turn through the router and return the full response + latency."""
    print(f'\n{"="*60}')
    if label:
        print(f'[{label}]')
    print(f'USER: {message}')
    print('ASSISTANT:', end=' ', flush=True)
    
    start = time.time()
    chunks = []
    for chunk in router.route_stream(message):
        chunks.append(chunk)
        # Print visible content, skip control sentinels
        if not chunk.startswith('<<'):
            print(chunk, end='', flush=True)
    elapsed = time.time() - start
    
    full = ''.join(chunks)
    sentinels = [c for c in chunks if c.startswith('<<')]
    content = full
    for s in ['<<CRITIC>>', '<<PREF_SAVING>>', '<<LOGISTICS>>']:
        content = content.replace(s, '')
    content = content.strip()
    
    print(f'\n[latency: {elapsed:.1f}s | sentinels: {sentinels}]')
    return content, sentinels, elapsed

## Scenario 1: Gift search with no constraints

In [ ]:
router = Router(customer_id=f'e2e_test_{uuid.uuid4().hex[:6]}')

content, sentinels, elapsed = run_conversation(
    router,
    'Can you recommend a flower bouquet for my wife\'s birthday?',
    label='Scenario 1: Gift search, no constraints'
)

assert len(content) > 50, 'Response should be substantive'
assert '<<CRITIC>>' not in sentinels, 'Critic should not engage when no constraints exist'
print('\nScenario 1: PASSED')

## Scenario 2: Allergen-aware search

In [ ]:
router = Router(customer_id=f'e2e_test_{uuid.uuid4().hex[:6]}')

# Turn 1: Establish the allergy
content1, sentinels1, _ = run_conversation(
    router,
    'My wife is allergic to nuts. Can you find her a birthday cake?',
    label='Scenario 2: Allergen-aware search (turn 1)'
)

assert '<<PREF_SAVING>>' in sentinels1, 'Preference should be saved'
# Critic should have engaged because of allergen constraint
print(f'Critic engaged: {"<<CRITIC>>" in sentinels1}')

# Response should NOT mention nuts or walnut products as recommendations
# (deterministic filtering should have removed them)
content_lower = content1.lower()
# This is a loose check — the LLM sometimes mentions allergens to warn about them
print(f'Response snippet: {content1[:300]}')

print('\nScenario 2: PASSED')

## Scenario 3: Delivery feasibility check

In [ ]:
router = Router(customer_id=f'e2e_test_{uuid.uuid4().hex[:6]}')

# Tier 1 — should get next-day delivery response
content, _, _ = run_conversation(
    router, 'Can you deliver to Colombo?',
    label='Scenario 3a: Tier 1 delivery'
)
assert 'colombo' in content.lower() or 'next' in content.lower() or 'day' in content.lower(), \
    f'Expected delivery info for Colombo. Got: {content[:200]}'

# Not covered district
content2, _, _ = run_conversation(
    router, 'Can you deliver to Kilinochchi?',
    label='Scenario 3b: Not-covered district'
)
not_covered_keywords = ['not', 'unavailable', 'unable', 'sorry', 'cover']
assert any(kw in content2.lower() for kw in not_covered_keywords), \
    f'Expected no-coverage message. Got: {content2[:200]}'

print('\nScenario 3: PASSED')

## Scenario 4: Mixed intent (search + logistics + preference update)

In [ ]:
router = Router(customer_id=f'e2e_test_{uuid.uuid4().hex[:6]}')

content, sentinels, elapsed = run_conversation(
    router,
    'My wife is allergic to nuts — find me a cake and can you deliver to Kandy by Friday?',
    label='Scenario 4: Mixed intent'
)

print(f'\nSentinels fired: {sentinels}')
print(f'Total latency  : {elapsed:.1f}s')

assert '<<PREF_SAVING>>' in sentinels, 'PREFERENCE_UPDATE should fire'
assert len(content) > 50, 'Response should cover both search and logistics'

print('\nScenario 4: PASSED')

## Scenario 5: Multi-turn — profile persists across turns

In [ ]:
import tempfile, json as _json
from memory.semantic_memory import SemanticMemory

# Use a temp profile file so we don't pollute production data
tmp = tempfile.NamedTemporaryFile(suffix='.json', delete=False, mode='w')
_json.dump({}, tmp)
tmp.close()

cid = f'e2e_mt_{uuid.uuid4().hex[:6]}'

# Turn 1: Save allergy
router1 = Router(customer_id=cid)
c1, s1, _ = run_conversation(router1, 'My wife is allergic to nuts.', label='Multi-turn turn 1')
assert '<<PREF_SAVING>>' in s1

# Turn 2: New router instance (simulates new session) — allergy should be remembered
router2 = Router(customer_id=cid)
c2, s2, _ = run_conversation(router2, 'Find a cake for my wife.', label='Multi-turn turn 2 (new session)')

print(f'\nTurn 2 sentinels: {s2}')
print(f'Turn 2 response (first 300 chars): {c2[:300]}')

# Critic should engage because the persisted allergy was loaded
# (this may or may not fire depending on whether profile was loaded correctly)
print('Multi-turn profile persistence test completed. Review output above.')
print('\nScenario 5: COMPLETED')

os.unlink(tmp.name)

## Scenario 6: Latency benchmark

In [ ]:
import statistics

queries = [
    ('Simple search, no constraints', 'Find a gift for my friend.', False),
    ('Search with allergen constraint', 'Find a cake for my wife who is allergic to nuts.', True),
    ('Logistics only', 'Can you deliver to Colombo tomorrow?', False),
]

latencies = {}

for label, query, has_constraint in queries:
    router = Router(customer_id=f'bench_{uuid.uuid4().hex[:6]}')
    start = time.time()
    chunks = list(router.route_stream(query))
    elapsed = time.time() - start
    latencies[label] = elapsed
    print(f'{label}: {elapsed:.1f}s')

print('\n=== Latency Summary ===')
for label, lat in latencies.items():
    print(f'  {label}: {lat:.1f}s')

# Sanity: all responses within 60 seconds
for label, lat in latencies.items():
    assert lat < 60, f'{label} took too long: {lat:.1f}s'

print('\nLatency benchmark: PASSED')

## Scenario 7: Short-term memory accumulates across turns

In [ ]:
router = Router(customer_id=f'e2e_stm_{uuid.uuid4().hex[:6]}')

run_conversation(router, 'Hi, I need gift ideas.', label='STM turn 1')
run_conversation(router, 'What about flowers?', label='STM turn 2')
run_conversation(router, 'Something under RS.2000?', label='STM turn 3')

# Check short-term memory accumulated
history = router.st_memory.get_history()
print(f'\nMessages in ST memory: {len(history)}')
for m in history:
    print(f"  [{m['role']}]: {m['content'][:80]}")

assert len(history) >= 4, f'Expected at least 4 messages (2 turns), got {len(history)}'
print('\nScenario 7 (STM accumulation): PASSED')